# 2 Mark5.3 본실험 70회 — 검증 결합기 가짜 정답(`--preset cmb`)

**L4 로 실행하십시오.** [셀 1] 이 Blackwell(sm_100·sm_120)이면 멈춥니다. GPU 가 없으면 CPU 로 돕니다(시간 제한 없음).

**'모두 실행' 한 번으로 끝까지 돕니다.** 세션이 끊기면 다시 '모두 실행' — 끝난 회차는 건너뜁니다.
`1 교사 점수·채택 관문` 노트북보다 먼저 시작해도 됩니다. [셀 5] 가 관문 결과(지금 점수 캐시로 만든 것)를 기다립니다.

| 순서 | 하는 일 |
|---|---|
| 1 | 관문 판정 읽기 — '채택(B+)' 이면 mn10 교사로 진행, 아니면 멈춤(함께 정함) |
| 2 | 결합기 가짜 정답 — `model/make_combiner_labels.py --scores plots/teacher_scores_mn10.csv` → `pseudo_labels_{cmb,cmbone,fix}_l<n|all>_s<seed>.csv` |
| 3 | 해시 사슬 확인 — `model/label_freshness.py`(교사 → 점수 캐시 → 결합기 CSV, 어긋나면 멈춤) |
| 4 | 학습 70회 — `model/run_experiments.py --mark_version mark5.2 --preset cmb --seeds 1 2 3 4 5` (회차마다 학습 → val → test) |
| 5 | 집계 — `model/aggregate_results.py --split test --main_pseudo_source cmb --bases <본실험 설정>` |
| 6 | 논문 표 — 비교모델 24회(`3 비교모델 결합기 조건 24회`)가 끝나길 기다렸다가 `make_paper_tables.py --main_pseudo_source cmb --baseline_cond plc --baseline_raw …` |

- 비교모델과의 짝 검정은 돌리지 않습니다. 비교모델은 조건마다 seed 1 한 번이라, seed 로 짝지으면 짝이 1개입니다. 검정 방식은 따로 정합니다.

**70회 구성 (14조합 × seed 1~5)** — 본실험 4(정답 전량·1,000·500·200 × 라벨 없는 2,000) · 라벨 없는 양 곡선 3(정답 200 × 0·500·1,000) · 지도만 1(정답 전량 × 0) · 아블레이션 6(두 후보 끔 `_noTWO` · 옛 교사 경로 `_TC` · 고정 규칙 `_psfix`, 각각 정답 200·전량).

- 학생 헤드는 선형 헤드라 제로샷은 없습니다. 옵티마이저는 Prodigy(run_tag `_PD`).
- 아블레이션 6조합의 표는 이 노트북이 만들지 않습니다(본표·라벨 효율 표만). 결과 파일은 `plots/` 에 남습니다.

In [ ]:
#========================================================================
# [셀 1] 런타임 확인 — L4 권장 · Blackwell(sm_100·120)이면 멈춤 · CPU 는 허용(멈추지 않음)
#========================================================================
import subprocess, sys
_probe = subprocess.run([sys.executable, "-c",
    "import os, torch\n"
    "ok = torch.cuda.is_available()\n"
    "cap = torch.cuda.get_device_capability(0) if ok else None\n"
    "print('GPU', torch.cuda.get_device_name(0) if ok else '없음 — CPU 로 돈다')\n"
    "print('SM', cap[0] * 10 + cap[1] if cap else 'cpu')\n"
    "print('torch', torch.__version__, '| CUDA', torch.version.cuda, '| CPU 코어', os.cpu_count())"],
    capture_output=True, text=True)
print(_probe.stdout, _probe.stderr[-2000:])
_sm = next((l.split()[1] for l in _probe.stdout.splitlines() if l.startswith("SM ")), "cpu")
if _sm in ("100", "120"):
    raise SystemExit("[STOP] Blackwell(sm_100/sm_120) 런타임입니다. L4 로 바꿔 다시 '모두 실행' 하십시오(2026-10-17 G4 계산 오류).")
!nvidia-smi --query-gpu=name,driver_version,compute_cap,memory.total --format=csv 2>/dev/null || echo "nvidia-smi 없음(CPU 런타임)"
!free -g | head -2

In [ ]:
#========================================================================
# [셀 2] 구글 드라이브 마운트
#========================================================================
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#========================================================================
# [셀 3] 작업 폴더 · 데이터 목록 · 드라이브 코드가 2026-10-23 판인가
#========================================================================
import csv, hashlib, json, os, re, time
EXP = "/content/drive/MyDrive/NewProject/codes/experiments"
ROOT = f"{EXP}/Mark5.3"
if not os.path.isdir(ROOT):
    raise SystemExit(f"[STOP] 코드 폴더가 없습니다: {ROOT}")
os.chdir(ROOT)
sys.path.insert(0, f"{ROOT}/model")


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


for name, n_expected in {"dataset_labeled.csv": 2000, "dataset_val.csv": 430,
                         "dataset_unlabeled.csv": 2000, "dataset_test.csv": 430}.items():
    if not os.path.isfile(name):
        raise SystemExit(f"[STOP] 데이터 목록이 없습니다: {name}")
    with open(name, newline="", encoding="utf-8-sig") as f:
        n = sum(1 for _ in csv.DictReader(f))
    print(name, n)
    if n != n_expected:
        raise SystemExit(f"[STOP] {name} 행 수 {n}, 기대 {n_expected}")
NEED = {"model/import_teachers.py": ['choices=["segment", "mn10"]'],
        "model/make_teacher_scores.py": ['"--teacher_kind"'],
        "model/teacher_gate_report.py": ["C4_raise_off_mean"],
        "model/make_combiner_labels.py": ['"--scores"'],
        "model/label_freshness.py": ["def check_scores", "def check_pseudo_csv"],
        "model/run_experiments.py": ["CMB_PLAN = [", 'choices=["paper", "plsource", "pl4", "cmb"]'],
        "model/aggregate_results.py": ['"--main_pseudo_source"'],
        "model/make_paper_tables.py": ['"--main_pseudo_source"', '"plc"'],
        "model/verify_release.py": ['"--refresh_source"']}
miss = [(f, s) for f, ss in NEED.items() for s in ss
        if not os.path.isfile(f) or s not in open(f, encoding="utf-8").read()]
if miss:
    raise SystemExit(f"[STOP] 드라이브 코드가 10-23 판이 아닙니다: {miss}")
size = os.path.getsize("weights/mn10_as_mAP_471.pt") if os.path.isfile("weights/mn10_as_mAP_471.pt") else None
print("mn10 가중치", size, "bytes (기대 19708753)")
if size != 19708753:
    raise SystemExit("[STOP] weights/mn10_as_mAP_471.pt 가 없거나 크기가 다릅니다")
print("코드 판 확인 완료")

In [ ]:
#========================================================================
# [셀 4] 패키지 — 코랩의 GPU torch·torchvision 은 그대로 둔다 · 문장 인코더 네 패키지만 판 고정
#========================================================================
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps",
                "sentence-transformers==2.7.0", "transformers==4.46.3",
                "huggingface-hub==0.36.2", "tokenizers==0.20.3"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "librosa==0.10.2.post1", "soundfile==0.12.1",
                "pydub==0.25.1", "thop==0.1.1.post2209072238"], check=True)
import sentence_transformers, transformers, huggingface_hub, tokenizers
got = {"sentence-transformers": sentence_transformers.__version__, "transformers": transformers.__version__,
       "huggingface-hub": huggingface_hub.__version__, "tokenizers": tokenizers.__version__}
want = {"sentence-transformers": "2.7.0", "transformers": "4.46.3", "huggingface-hub": "0.36.2", "tokenizers": "0.20.3"}
print(got)
if got != want:
    raise SystemExit(f"[STOP] 패키지 판 불일치 {got} — 런타임 → 세션 다시 시작 → 다시 '모두 실행'")
import shutil
from pathlib import Path
HF_SRC = Path("/content/drive/MyDrive/NewProject/hf_cache/models--sentence-transformers--all-MiniLM-L6-v2")
HF_HUB = Path.home() / ".cache" / "huggingface" / "hub"
if not HF_SRC.is_dir():
    raise SystemExit(f"[STOP] 문장 인코더 캐시가 없습니다: {HF_SRC}")
HF_HUB.mkdir(parents=True, exist_ok=True)
shutil.copytree(HF_SRC, HF_HUB / HF_SRC.name, dirs_exist_ok=True)
os.environ.update({"HF_HOME": str(HF_HUB.parent), "HF_HUB_CACHE": str(HF_HUB),
                   "HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1"})
from sentence_transformers import SentenceTransformer
SentenceTransformer("all-MiniLM-L6-v2")
print("문장 인코더 오프라인 캐시 준비 완료")

In [ ]:
#========================================================================
# [셀 5] 관문 판정 읽기 — 지금 점수 캐시로 만든 관문 결과가 나올 때까지 기다린다(시간 제한 없음)
#========================================================================
ADOPTED_OVERRIDE = None   # 관문이 '채택(B+)' 이 아니어서 함께 정한 경우에만 적는다. 기본 None = 관문 판정을 따른다
GATE = "plots/teacher_gate_mn10_vs_segment.json"


def gate_now():
    """관문 json 이 있고, 그 안의 두 점수 캐시 해시가 지금 파일과 같으면 json, 아니면 None."""
    if not os.path.isfile(GATE):
        return None
    g = json.load(open(GATE, encoding="utf-8"))
    for side, kind in (("candidate", "mn10"), ("reference", "segment")):
        p = f"plots/teacher_scores_{kind}.csv"
        if not os.path.isfile(p) or g[side].get("sha256") != sha256(p):
            return None
    return g


t0 = time.time()
while (g := gate_now()) is None:
    print(time.strftime("%H:%M:%S"), f"(경과 {(time.time() - t0) / 60:.0f}분) 관문 결과를 기다리는 중 — '1 교사 점수·채택 관문' 노트북", flush=True)
    time.sleep(60)
print(f"[관문] {g['n_satisfied']}/4 → {g['verdict']} · 후보 캐시 {g['candidate']['sha256'][:12]} · 기준 캐시 {g['reference']['sha256'][:12]}")
ADOPTED = ADOPTED_OVERRIDE or ("mn10" if g["verdict"] == "채택(B+)" else None)
if ADOPTED is None:
    raise SystemExit("[STOP] 관문이 '채택(B+)' 이 아닙니다 — 결과를 보고 어느 교사를 쓸지 함께 정한 뒤 ADOPTED_OVERRIDE 를 적고 다시 '모두 실행'")
if ADOPTED != "mn10":
    raise SystemExit("[STOP] B0(segment)로 갈 때는 _TC 아블레이션(mn10 특징 캐시로 고정)을 어떻게 할지 코드에서 먼저 정해야 합니다")
SCORES = f"plots/teacher_scores_{ADOPTED}.csv"
print("가짜 정답을 만들 교사:", ADOPTED, "|", SCORES)

In [ ]:
#========================================================================
# [셀 6] 결합기 가짜 정답 60개(cmb·cmbone·fix × 정답 200·500·1,000·전량 × seed 1~5) — 최신 판이면 건너뛴다
#========================================================================
import label_freshness
EXPECTED = [f"pseudo_labels_{ps}_l{l}_s{s}.csv" for ps in ("cmb", "cmbone", "fix")
            for l in (200, 500, 1000, "all") for s in (1, 2, 3, 4, 5)]


def stale_labels():
    bad = []
    for name in EXPECTED:
        try:
            meta = label_freshness.check_pseudo_csv(name)
            if meta["scores_sha256"] != sha256(SCORES):
                bad.append(f"{name}: 채택한 교사({ADOPTED})가 아닌 점수 캐시로 만든 판")
        except label_freshness.StaleFileError as e:
            bad.append(f"{name}: {e}")
    return bad


bad = stale_labels()
if bad:
    print(f"다시 만듭니다({len(bad)}개 어긋남, 예: {bad[0]})")
    subprocess.run([sys.executable, "-u", "model/make_combiner_labels.py", "--scores", SCORES], check=True)
    bad = stale_labels()
    if bad:
        raise SystemExit(f"[STOP] 결합기 가짜 정답을 만든 뒤에도 최신 판 확인을 통과하지 못했습니다: {bad[:5]}")
else:
    print("결합기 가짜 정답 60개가 지금 교사의 최신 판입니다 — 건너뜁니다.")
subprocess.run([sys.executable, "model/label_freshness.py"], check=True)

In [ ]:
#========================================================================
# [셀 7] 실행 계획 — 70회인가 · 옛 판 파일이 없는가 (아무것도 안 돌림)
#========================================================================
RUN = "--mark_version mark5.2 --preset cmb --seeds 1 2 3 4 5"


def plan():
    p = subprocess.run(f"{sys.executable} model/run_experiments.py {RUN} --dry_run", shell=True, capture_output=True, text=True)
    m = re.search(r"완료 (\d+)개 / 남은 (\d+)개", p.stdout)
    if p.returncode != 0 or not m:
        print(p.stdout[-3000:]); print(p.stderr[-3000:])
        raise SystemExit("[STOP] 실행기 계획(--dry_run)을 읽지 못했습니다 — 위 출력 확인")
    if "[STALE]" in p.stdout:
        print(p.stdout[-3000:])
        raise SystemExit("[STOP] 옛 판 가짜 정답·교사 캐시가 있습니다")
    todo = [l.strip() for l in p.stdout.splitlines() if "[실행]" in l]
    tags = re.findall(r"\[(?:실행|건너뜀)\][^\n]*?\s(mark5\.2_\S+)", p.stdout)
    return int(m.group(1)), int(m.group(2)), todo, tags


done, left, todo, tags = plan()
print(f"계획 {len(tags)}회 (서로 다른 태그 {len(set(tags))}) · 완료 {done} · 남은 {left}")
if len(tags) != 70 or len(set(tags)) != 70 or done + left != 70:
    raise SystemExit(f"[STOP] 계획이 70회가 아닙니다({len(tags)}회 · 서로 다른 태그 {len(set(tags))})")
if any("_PD" not in t for t in tags):
    raise SystemExit("[STOP] Prodigy(_PD) 가 아닌 태그가 있습니다")

In [ ]:
#========================================================================
# [셀 8] 본실험 ★ 70회 — 회차마다 학습 → val → test. 끝난 회차는 건너뛴다. 시간 제한 없음
#========================================================================
!python model/run_experiments.py {RUN}

In [ ]:
#========================================================================
# [셀 9] 확인 — 70회가 전부 끝났는가 (남았으면 멈춘다)
#========================================================================
done, left, todo, tags = plan()
if left:
    print("\n".join(todo[:20]))
    raise SystemExit(f"[STOP] {left}회가 끝나지 않았습니다 — logFiles/<태그>__*.log 확인 뒤 다시 '모두 실행'(끝난 회차는 건너뜁니다)")
print("70회 전부 완료")

In [ ]:
#========================================================================
# [셀 10] 집계(Mark5.3 만) — 본표 · 라벨 효율 표 (아블레이션 설정은 --bases 로 뺀다)
#========================================================================
import run_experiments as RE
import aggregate_results as AG
_main_kw = RE._parse_runner_extra(RE.CMB_BASE_EXTRA)[1]
MAIN_BASE = AG._parse_tag_full(RE._run_tag("mark5.2", 1, 2000, labeled_n=None, ps="cmb", ablation_kw=_main_kw))[0]
print("본실험 설정:", MAIN_BASE)
subprocess.run([sys.executable, "-u", "model/aggregate_results.py", "--split", "test",
                "--main_pseudo_source", "cmb", "--bases", MAIN_BASE], check=True)

In [ ]:
#========================================================================
# [셀 11] 논문 표 — 비교모델 3개 × 8회(지도만 4 · 결합기 한 라벨 4)가 끝나길 기다린 뒤 표에 비교모델 행을 붙인다 (시간 제한 없음)
#========================================================================
DATA_ROOT = f"{EXP}/Mark5.2"
BASELINES = {"Baseline_EfficientAT_mn10": "efficientat_mn10", "Baseline_PANNs_CNN6": "panns_cnn6", "Baseline_CED_Tiny": "ced_tiny"}


def baseline_left(repo):
    p = subprocess.run(f"{sys.executable} run_experiments.py --preset cmb --data_root {DATA_ROOT} --dry_run",
                       shell=True, capture_output=True, text=True, cwd=f"{EXP}/{repo}")
    rows = [l for l in p.stdout.splitlines() if "[실행]" in l or "[건너뜀]" in l]
    if p.returncode != 0 or len(rows) != 8:
        return None
    return sum("[실행]" in l for l in rows)


t0 = time.time()
while True:
    left = {repo: baseline_left(repo) for repo in BASELINES}
    if all(v == 0 for v in left.values()):
        break
    print(time.strftime("%H:%M:%S"), f"(경과 {(time.time() - t0) / 60:.0f}분) 비교모델 남은 회차:", left, flush=True)
    time.sleep(300)
raws = []
for repo, mtag in BASELINES.items():
    subprocess.run([sys.executable, "aggregate_baseline.py", "--split", "test", "--conds", "sup", "plc", "--seeds", "1"],
                   cwd=f"{EXP}/{repo}", check=True)
    raws.append(f"{EXP}/{repo}/plots/aggregate_raw_{mtag}_test.csv")
# 짝 검정(aggregate_results --baseline_raw)은 돌리지 않는다: 비교모델은 seed 1 하나뿐이라 seed 로 짝지으면 짝이 1개다(검정 방식 미정).
subprocess.run([sys.executable, "-u", "model/make_paper_tables.py", "--split", "test", "--main_pseudo_source", "cmb",
                "--baseline_cond", "plc", "--baseline_raw", *raws], check=True)
print("완료 — 표는 plots/ 의 paper 표 파일들")